# Building and Automating an ETL Notebook in Databricks

In this notebook we will build a complete ETL pipeline — loading raw weather data, transforming it, and persisting curated results as a Delta table. Once built, this notebook can be scheduled as a Databricks Job to run automatically on a cadence.

**Components:**
1. Load the data
2. Transform the data
3. Persist the data
4. Review table transaction logs

## Setup
You need to enter the name of a catalog and schema that you have access onto so that tables and views can be created

In [0]:
catalog_to_write_to = "databricks_day"
schema_to_write_to = "weather"

write_path = f"{catalog_to_write_to}.{schema_to_write_to}"

## 1. Load the Data

There are several ways to bring data into a notebook. We'll cover three common patterns:
- **SQL temp view** — query Unity Catalog tables using SQL
- **PySpark DataFrame** — read a table directly into Python
- **File-based** — read CSV or Excel from a Databricks Volume

**Option 1: Create a Temp View and Query with SQL**

Temp views exist only for the duration of the session. They're useful when you want to break SQL work into reusable chunks without writing anything to the catalog.

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW temp_view AS
SELECT * FROM accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric;

SELECT * FROM temp_view LIMIT 5;

**Option 2: Read a Unity Catalog Table into a PySpark DataFrame**

Reading directly into a DataFrame gives you the full power of PySpark for transformations. `spark.table()` takes a three-level Unity Catalog name: `catalog.schema.table`.

In [0]:
df = spark.table("accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric")

display(df.limit(5))

**Option 3a: Read a CSV from a Databricks Volume**

Databricks Volumes provide governed storage for files. You can upload CSV, Excel, JSON, and other files to a Volume and read them directly in a notebook using the `/Volumes/` path prefix.

In [0]:
volume_path = f"/Volumes/{catalog_to_write_to}/{schema_to_write_to}/files"

df = spark.read.csv(f"{volume_path}/weather.csv", header=True, inferSchema=True)

display(df.limit(5))

**Option 3b: Read an Excel file from a Databricks Volume**

Excel files can also be read from Volumes. Databricks handles the parsing — no extra library installation needed.

In [0]:
df = spark.read.excel(f"{volume_path}/weather.xlsx")

display(df.limit(5))

## 2. Transform the Data

Using PySpark we'll clean and aggregate the raw data into a curated dataset. We'll work through three common transformation steps: selecting relevant columns, removing bad data, and aggregating to a useful grain.

**Start with a fresh copy of the source data**

It's good practice to reload from the source at the start of your transformation chain so the notebook is idempotent — it produces the same result every time it runs.

In [0]:
df = spark.table("accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric")

display(df.limit(5))

**Step 1: Select only the columns you need**

Dropping unused columns early reduces memory usage and keeps downstream logic easier to follow.

In [0]:
df = df.select("latitude", "longitude", "station_code", "date", "phrase_long", "temperature_avg", "wind_speed_avg", "precipitation_probability", "rain_lwe_total")

display(df.limit(5))

**Step 2: Remove bad data**

Drop rows with nulls and filter out known bad records. In a production pipeline you'd typically log the dropped rows rather than silently discard them.

In [0]:
df = df.dropna().filter(df.station_code != "NYCG")

display(df.limit(5))

**Step 3: Aggregate to monthly averages**

Roll up the daily records to monthly averages per station. This is the curated grain we'll persist as a Delta table.

In [0]:
from pyspark.sql.functions import month, avg

df_monthly_avg = df.withColumn("month", month("date")) \
    .groupBy("station_code", "month") \
    .agg(
        avg("temperature_avg").alias("avg_temperature"),
        avg("wind_speed_avg").alias("avg_wind_speed"),
        avg("precipitation_probability").alias("avg_precip_probability"),
        avg("rain_lwe_total").alias("avg_rain_lwe")
    )

display(df_monthly_avg.limit(5))

## 3. Persist the Data

After transforming the data, we need to write it somewhere durable so other users and tools can access it. There are three main patterns:

| Pattern | When to use |
|---|---|
| **View** | Ad-hoc / lightweight — no data stored, transformation re-runs on every query |
| **Overwrite table** | Full refresh — simplest option, replaces the entire table each job run |
| **Delta merge (upsert)** | Incremental updates — adds new rows and updates changed ones, most production-ready |

**Option 1: Create a View**

A view stores the SQL definition, not the data. Every query re-runs the transformation against the source table — useful for always-fresh, ad-hoc datasets. For scheduled jobs that serve downstream users, a persisted table is usually a better choice.

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {write_path}.monthly_avg_view AS
SELECT
    station_code,
    MONTH(date) AS month,
    AVG(temperature_avg) AS avg_temperature,
    AVG(wind_speed_avg) AS avg_wind_speed,
    AVG(precipitation_probability) AS avg_precip_probability,
    AVG(rain_lwe_total) AS avg_rain_lwe
FROM accuweather_daily_and_hourly_forecasts_u_s_postal_codes_sample.forecast.us_postal_daily_metric
GROUP BY station_code, MONTH(date)
""")

display(spark.sql(f"SELECT * FROM {write_path}.monthly_avg_view LIMIT 5"))

**Option 2: Overwrite — Write a Full Table on Every Run**

The simplest persistence pattern: replace the entire table each time the job runs. Works well when the source dataset is small enough to fully reprocess and you don't need to preserve history.

In [0]:
df_monthly_avg.write.mode("overwrite").saveAsTable(f"{write_path}.weather_monthly_avg")

df = spark.table(f"{write_path}.weather_monthly_avg")

display(df.limit(5))

**Option 3: Delta Merge — Upsert New and Changed Rows**

When new data arrives incrementally, a merge (upsert) is more efficient than a full overwrite. Delta's merge operation matches records on a key, updates rows that already exist, and inserts rows that are new.

In [0]:
from pyspark.sql import Row
from delta.tables import DeltaTable

# Simulating a batch of incoming new data — in a real pipeline this would come from
# a source table or file that arrives on each job run.
new_data = [
    Row(station_code="ORNY", month=4, avg_temperature=14.174545, avg_wind_speed=2.803636, avg_precip_probability=17.90909090909091, avg_rain_lwe=0.078509091),
    Row(station_code="ABC",  month=4, avg_temperature=15.132727, avg_wind_speed=2.845455, avg_precip_probability=41.81818181818182, avg_rain_lwe=2.496131818)
]

df_new = spark.createDataFrame(new_data)

display(df_new)

In [0]:
delta_table = DeltaTable.forName(spark, f"{write_path}.weather_monthly_avg")

delta_table.alias("target").merge(
    df_new.alias("source"),
    "target.station_code = source.station_code AND target.month = source.month"
).whenMatchedUpdate(set={
    "avg_temperature": "source.avg_temperature",
    "avg_wind_speed": "source.avg_wind_speed",
    "avg_precip_probability": "source.avg_precip_probability",
    "avg_rain_lwe": "source.avg_rain_lwe"
}).whenNotMatchedInsert(values={
    "station_code": "source.station_code",
    "month": "source.month",
    "avg_temperature": "source.avg_temperature",
    "avg_wind_speed": "source.avg_wind_speed",
    "avg_precip_probability": "source.avg_precip_probability",
    "avg_rain_lwe": "source.avg_rain_lwe"
}).execute()

display(spark.table(f"{write_path}.weather_monthly_avg").limit(5))

## 4. Delta Transaction Logs

Every write to a Delta table is recorded in a transaction log. This gives you a full audit trail of every change — who wrote what, when, and using which operation (overwrite, merge, etc.).

You can also use this log to **time-travel**: query the table as it looked at any previous version.

In [0]:
delta_table = DeltaTable.forName(spark, f"{write_path}.weather_monthly_avg")

display(delta_table.history().limit(5))

**Time Travel — Query a Previous Version**

You can read any previous version of the table using `VERSION AS OF` in SQL, or the `.option("versionAsOf", ...)` reader option in PySpark.

In [0]:
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table(f"{write_path}.weather_monthly_avg")

display(df_v0.limit(5))

---

## 5. Challenge Activity

**Goal:** Build your own mini ETL pipeline from scratch using what you've learned.

You're going to answer the question: **"Which weather stations have the highest rain risk?"**

Define "rain risk" as stations where the average precipitation probability exceeds a threshold you choose, and rank them.

### Your pipeline should:
1. **Load** the raw AccuWeather source data into a DataFrame
2. **Transform** — select useful columns, clean nulls, and aggregate to find the average precipitation probability and average rain total **per station**
3. **Persist** — write the result to a new table called `weather_rain_risk` in your catalog/schema
4. **Verify** — query the table, order by highest rain risk, and display the top 10 stations

### Stretch goal:
- Add a column `risk_tier` that labels each station `"High"`, `"Medium"`, or `"Low"` based on the precipitation probability value
- Write the result back to the table using a Delta merge instead of overwrite

**Useful hints are in the cells below. Fill in the `# TODO` sections.**

**Step 1: Load the data**

In [0]:
# TODO: Load the AccuWeather source table into a DataFrame called df_challenge
# Hint: spark.table("catalog.schema.table")

df_challenge = # TODO

display(df_challenge.limit(5))

**Step 2: Transform — clean and aggregate**

In [0]:
from pyspark.sql.functions import avg, desc

# TODO: Select the columns you need — at minimum: station_code, precipitation_probability, rain_lwe_total
# Hint: df.select("col1", "col2", ...)
df_challenge = df_challenge.select(# TODO
)

# TODO: Drop rows with nulls
df_challenge = # TODO

# TODO: Aggregate by station_code — compute average precipitation_probability and average rain_lwe_total
# Hint: .groupBy("station_code").agg(avg("col").alias("new_name"), ...)
df_rain_risk = df_challenge.groupBy(# TODO
).agg(
    # TODO
)

display(df_rain_risk.orderBy(desc("avg_precip_probability")).limit(10))

**Step 3: Persist — write to a new Delta table**

In [0]:
# TODO: Write df_rain_risk to a new table called "weather_rain_risk" in your catalog/schema
# Hint: df.write.mode("overwrite").saveAsTable(f"{write_path}.table_name")

# TODO

**Step 4: Verify — query the table and show the top 10 rainiest stations**

In [0]:
# TODO: Read the table back and display the top 10 stations ordered by highest avg_precip_probability
# Hint: spark.table(...).orderBy(desc("col")).limit(10)

df_verify = # TODO

display(df_verify)

---

### Stretch Goal: Add a `risk_tier` column and merge

Classify each station into `"High"` (avg precipitation probability ≥ 50%), `"Medium"` (25–50%), or `"Low"` (< 25%) and write back using Delta merge.

In [0]:
from pyspark.sql.functions import when, col

# Add a risk_tier label based on avg_precip_probability
df_tiered = df_rain_risk.withColumn(
    "risk_tier",
    when(col("avg_precip_probability") >= 50, "High")
    .when(col("avg_precip_probability") >= 25, "Medium")
    .otherwise("Low")
)

display(df_tiered.orderBy(desc("avg_precip_probability")).limit(10))

In [0]:
# TODO: Use Delta merge to upsert df_tiered into weather_rain_risk, matching on station_code
# Hint: refer back to the merge pattern in Section 3, Option 3 above

# TODO

**Verify the transaction log for your new table**

In [0]:
challenge_table = DeltaTable.forName(spark, f"{write_path}.weather_rain_risk")

display(challenge_table.history())